# Ticket 3: chart of accounts + account mapping

The three checks that decided the mapping in `docs/mapping-review.md`:
whether the "clean" suspense/clearing accounts are actually still in
use, whether the "immaterial" accounts are actually zero, and whether
`199999`/`999999` are real accounts or migration catch-all codes. See
`docs/missions/01-chart-of-accounts-mapping.md` for the full mapping and
`docs/adr/0005-account-status-catch-all.md` for the catch-all status.

In [ ]:
import duckdb # type: ignore

con = duckdb.connect("../warehouse.duckdb", read_only=True)
SCOPE = "company_code = 1000 AND fiscal_year = 2024 AND fiscal_period IN (1,2,3)"

## The 5 "clean" suspense/clearing accounts: still live?

The first-pass report proposed `status=deprecated` for these 5 because
each has one clear description ("General Suspense", "Payroll Clearing",
etc). Checking the full dataset instead of just the current scope told a
different story.

In [2]:
for acc in (9000, 9100, 9300, 199000, 199300):
    r = con.execute(f"""
        SELECT COUNT(*), COUNT(DISTINCT company_code), COUNT(DISTINCT (fiscal_year, fiscal_period)),
               ROUND(SUM(local_amount),2), MIN((fiscal_year,fiscal_period)), MAX((fiscal_year,fiscal_period))
        FROM stg_gl WHERE gl_account = {acc}
    """).fetchone()
    print(acc, 'lines=', r[0], 'companies=', r[1], 'periods=', r[2], 'net_local=', r[3], 'range=', r[4], '-', r[5])

9000 lines= 3020 companies= 4 periods= 13 net_local= -20302850.63 range= (2024, 1) - (2025, 1)
9100 lines= 4986 companies= 4 periods= 13 net_local= -117093307.07 range= (2024, 1) - (2025, 1)
9300 lines= 3391 companies= 4 periods= 13 net_local= -30316168.34 range= (2024, 1) - (2025, 1)
199000 lines= 923 companies= 4 periods= 13 net_local= -25703341.02 range= (2024, 1) - (2025, 1)
199300 lines= 1950 companies= 4 periods= 13 net_local= 70001585.27 range= (2024, 1) - (2025, 1)


All 5 are active across all 4 companies and 13 periods, with material
balances, continuing into 2025-P01. Not dormant. Per
`docs/business-rules.md`'s own reading of "deprecated" (a design decision
about the target CoA, separate from operational status, see
`docs/definitions.md`), these 5 shipped as `status=mapped` with their own
code as target, `source_usage=live` — not `deprecated`.

## The 6 "immaterial" unmapped accounts: really $0?

The first-pass report showed `local_amount` netting to 0 for all six and
called them safe to leave unmapped. Checking `debit_amount`/`credit_amount`
instead of `local_amount` told a different story.

In [3]:
con.execute("""
    SELECT gl_account, ROUND(SUM(debit_amount),2), ROUND(SUM(credit_amount),2)
    FROM stg_gl WHERE gl_account IN (115020,115021,115030,205020,205021,205030)
    GROUP BY 1 ORDER BY 1
""").fetchall()

[(115020, 4354887.21, 0.0),
 (115021, 5460009.21, 0.0),
 (115030, 2326157.14, 0.0),
 (205020, 0.0, 4268147.68),
 (205021, 0.0, 6397397.79),
 (205030, 0.0, 3067126.1)]

`local_amount` is 0 on every single row for all six, but real money moves
through as three debit-only/credit-only pairs (`115020`\u2194`205020`,
`115021`\u2194`205021`, `115030`\u2194`205030`), millions per pair. Shipped
as `status=unmapped` (unchanged) but tagged `account_role=clearing_pair`,
`pair_id`, and `dq_flag=local_amount_zero_but_dr_cr_nonzero`, with the
underlying data defect filed on issue #5.

## 199999 / 999999: one account, or a catch-all code?

Each carries a dozen-plus unrelated `account_description` values on the
same `gl_account`. Checking whether that pattern holds beyond the current
scope is what led to ADR-0005.

In [4]:
for acc in (199999, 999999):
    r = con.execute(f"""
        SELECT COUNT(*), COUNT(DISTINCT company_code), COUNT(DISTINCT (fiscal_year, fiscal_period)),
               ROUND(SUM(local_amount),2)
        FROM stg_gl WHERE gl_account = {acc}
    """).fetchone()
    print(acc, 'lines=', r[0], 'companies=', r[1], 'periods=', r[2], 'net_local (all scope)=', r[3])

199999 lines= 2140 companies= 4 periods= 13 net_local (all scope)= 180162774.79
999999 lines= 1623 companies= 4 periods= 13 net_local (all scope)= 1736047.49


Both span all 4 companies and 13 periods, with `199999` carrying
180,162,775 net local overall against just 5,751,571 in the current
scope \u2014 most of the impact is outside the window this ticket is even
looking at. Matches SAP's own documented dummy/default-object pattern for
migration catch-all codes, not a real account.

In [5]:
# within the CURRENT scope, do these two look internally consistent?
for acc in (199999, 999999):
    r = con.execute(f"""
        SELECT financial_statement_category, COUNT(*) FROM stg_gl
        WHERE {SCOPE} AND gl_account = {acc} GROUP BY 1 ORDER BY 2 DESC
    """).fetchall()
    print(acc, '->', r)

199999 -> [('asset', 174)]
999999 -> [('suspense', 78)]


Within the current scope alone, each looks internally consistent (one
category each) \u2014 the inconsistency only shows up cross-scope. This is
why ADR-0005 forces `fs_category_flag=true` for `catch_all` accounts by
hardcode rather than relying on the same in-scope self-consistency check
used for every other account: the signal this flag exists to carry isn't
visible in this query window.

## What I've got

Three rulings in `map_account.csv`, none of them the first-pass answer:

1. `9000, 9100, 9300, 199000, 199300` → `status=mapped`, still live,
   not `deprecated`.
2. `115020/205020, 115021/205021, 115030/205030` → stay `unmapped`, now
   tagged `dq_flag`/`pair_id`/`account_role` instead of read as zero.
3. `199999, 999999` → new status `catch_all` (ADR-0005), forced-flagged
   rather than relying on scope-local self-consistency.